#ML Layer Training: Random Forest + XGBoost


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/kwago/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import re
import pickle
from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import nltk
from nltk.corpus import stopwords
from imblearn.over_sampling import SMOTE
import scipy.sparse as sp

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.utils.class_weight import compute_class_weight


nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

True

## Load Preprocessed Data

In [ ]:
with open(BASE + 'tfidf.pkl', 'rb') as f:   tfidf = pickle.load(f)
with open(BASE + 'scaler.pkl', 'rb') as f:  scaler = pickle.load(f)
with open(BASE + 'label_encoder.pkl', 'rb') as f: le = pickle.load(f)

X_train_A = sp.load_npz(BASE + 'X_train_A.npz')
X_val_A   = sp.load_npz(BASE + 'X_val_A.npz')
X_test_A  = sp.load_npz(BASE + 'X_test_A.npz')

y_train_A = np.load(BASE + 'y_train_A.npy')
y_val     = np.load(BASE + 'y_val.npy')
y_test    = np.load(BASE + 'y_test.npy')

print(f"X_train shape: {X_train_A.shape}, y_train shape: {y_train_A.shape}")
print(f"X_val shape: {X_val_A.shape}, y_val shape: {y_val.shape}")
print(f"X_test shape: {X_test_A.shape}, y_test shape: {y_test.shape}")

X_train shape: (12414, 5019), y_train shape: (12414,)
X_val shape: (1636, 5019), y_val shape: (1636,)
X_test shape: (1637, 5019), y_test shape: (1637,)


## Helper

In [ ]:
def evaluate(name, model, X, y, threshold=0.5):
    prob = model.predict_proba(X)[:, 1]
    pred = (prob >= threshold).astype(int)
    metrics = {
        'accuracy': accuracy_score(y, pred),
        'precision': precision_score(y, pred, zero_division=0),
        'recall': recall_score(y, pred, zero_division=0),
        'f1': f1_score(y, pred, zero_division=0),
        'auc_roc': roc_auc_score(y, prob),
    }
    print(f'\n=== {name} ===')
    for k, v in metrics.items():
        print(f'  {k:<12}: {v:.4f}')
    print(f'\n  Confusion Matrix:\n{confusion_matrix(y, pred)}')
    print(f'\n{classification_report(y, pred, target_names=["Benign","Phishing"], zero_division=0)}')
    return metrics

## Random Forest — Hyperparameter Tuning

In [ ]:

param_dist_rf = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [None, 10, 20, 30],
    'min_samples_leaf': [1, 2, 4, 8],
    'min_samples_split': [2, 5, 10],
    'criterion': ['gini', 'entropy'],
    'max_features': ['sqrt', 'log2'],
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=param_dist_rf,
    n_iter=50,
    scoring='f1',
    cv=cv,
    random_state=42,
    verbose=2,
    n_jobs=-1,
)
rf_search.fit(X_train_A, y_train_A)

print(f'\nBest RF params : {rf_search.best_params_}')
print(f'Best CV F1     : {rf_search.best_score_:.4f}')

NameError: name 'StratifiedKFold' is not defined

In [ ]:
rf_best = rf_search.best_estimator_
rf_val_metrics  = evaluate('RF Validation', rf_best, X_val_A, y_val)
rf_test_metrics = evaluate('RF Test',       rf_best, X_test_A, y_test)

## XGBoost — Hyperparameter Tuning


In [ ]:
param_dist_xgb = {
    'n_estimators': [100, 200, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'reg_alpha': [0, 0.1, 0.5, 1.0],
    'reg_lambda': [0.5, 1.0, 1.5, 2.0],
    'min_child_weight': [1, 3, 5],
}

xgb_search = RandomizedSearchCV(
    XGBClassifier(
        random_state=42,
        eval_metric='logloss',
        tree_method='hist',
        n_jobs=-1,
    ),
    param_distributions=param_dist_xgb,
    n_iter=50,
    scoring='f1',
    cv=cv,
    random_state=42,
    verbose=2,
    n_jobs=-1,
)
xgb_search.fit(X_train_A, y_train_A)

print(f'\nBest XGBoost params : {xgb_search.best_params_}')
print(f'Best CV F1          : {xgb_search.best_score_:.4f}')

In [ ]:
xgb_best = xgb_search.best_estimator_
xgb_val_metrics  = evaluate('XGBoost — Validation', xgb_best, X_val_A, y_val)
xgb_test_metrics = evaluate('XGBoost — Test',       xgb_best, X_test_A, y_test)

## ML Layer Ensemble — Weight + Threshold Optimization

In [ ]:
rf_val_prob  = rf_best.predict_proba(X_val_A)[:, 1]
xgb_val_prob = xgb_best.predict_proba(X_val_A)[:, 1]

best_f1, best_w_rf, best_threshold = 0.0, 0.5, 0.5

for w_rf in np.arange(0.0, 1.01, 0.05):
    w_xgb = 1.0 - w_rf
    ensemble_prob = w_rf * rf_val_prob + w_xgb * xgb_val_prob
    for thresh in np.arange(0.30, 0.71, 0.05):
        pred  = (ensemble_prob >= thresh).astype(int)
        score = f1_score(y_val, pred, zero_division=0)
        if score > best_f1:
            best_f1       = score
            best_w_rf     = round(float(w_rf), 2)
            best_threshold = round(float(thresh), 2)

print(f'Optimal RF weight     : {best_w_rf}')
print(f'Optimal XGBoost weight: {round(1 - best_w_rf, 2)}')
print(f'Optimal threshold     : {best_threshold}')
print(f'Val F1 at optimal     : {best_f1:.4f}')

### ML Layer — Final Test Set Evaluation

In [ ]:
rf_test_prob  = rf_best.predict_proba(X_test_A)[:, 1]
xgb_test_prob = xgb_best.predict_proba(X_test_A)[:, 1]

ensemble_test_prob = best_w_rf * rf_test_prob + (1 - best_w_rf) * xgb_test_prob
ensemble_test_pred = (ensemble_test_prob >= best_threshold).astype(int)

print('=== ML LAYER ENSEMBLE — TEST SET ===')
print(f'Accuracy : {accuracy_score(y_test, ensemble_test_pred):.4f}')
print(f'Precision: {precision_score(y_test, ensemble_test_pred, zero_division=0):.4f}')
print(f'Recall   : {recall_score(y_test, ensemble_test_pred, zero_division=0):.4f}')
print(f'F1-Score : {f1_score(y_test, ensemble_test_pred, zero_division=0):.4f}')
print(f'AUC-ROC  : {roc_auc_score(y_test, ensemble_test_prob):.4f}')
print(f'\nConfusion Matrix:\n{confusion_matrix(y_test, ensemble_test_pred)}')
print(f'\n{classification_report(y_test, ensemble_test_pred, target_names=["Benign","Phishing"], zero_division=0)}')

## Save Models and Artifacts

In [ ]:
# Random Forest
!pip install skl2onnx -q

from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import onnx

n_features = X_train_A.shape[1]
initial_type = [('float_input', FloatTensorType([None, n_features]))]

rf_onnx = convert_sklearn(
    rf_best,
    initial_types=initial_type,
    target_opset=17,
    options={type(rf_best): {'zipmap': False}},
)
onnx.save_model(rf_onnx, BASE + 'rf_model.onnx')


xgb_best.save_model(BASE + 'xgboost_model.onnx')


ml_weights = {
    'rf_weight'  : best_w_rf,
    'xgb_weight' : round(1 - best_w_rf, 2),
    'threshold'  : best_threshold,
    'val_f1'     : round(best_f1, 4),
}
with open(BASE + 'ml_layer_weights.json', 'w') as f:
    json.dump(ml_weights, f, indent=2)
print(f'Saved: ml_layer_weights.json → {ml_weights}')

# Summary CSV
results = pd.DataFrame([
    {'Model':'Random Forest',       'Set':'Test', **{k: round(v,4) for k,v in rf_test_metrics.items()}},
    {'Model':'XGBoost',             'Set':'Test', **{k: round(v,4) for k,v in xgb_test_metrics.items()}},
    {'Model':'ML Layer Ensemble',   'Set':'Test',
     'accuracy' : round(accuracy_score(y_test, ensemble_test_pred), 4),
     'precision': round(precision_score(y_test, ensemble_test_pred, zero_division=0), 4),
     'recall'   : round(recall_score(y_test, ensemble_test_pred, zero_division=0), 4),
     'f1'       : round(f1_score(y_test, ensemble_test_pred, zero_division=0), 4),
     'auc_roc'  : round(roc_auc_score(y_test, ensemble_test_prob), 4)},
])
results.to_csv(BASE + 'ml_layer_results.csv', index=False)
print('\nResults summary:')
print(results.to_string(index=False))